In [1]:
import json, pandas as pd
runs = [json.loads(l) for l in open('wk6_readability_dev.jsonl', encoding='utf-8')]
df = pd.DataFrame([{'question': r['question'], 'group': r['group'],
 **{k: v for k, v in r['scores'].items() if k != 'unsupported'}}
 for r in runs])
rag = df[(df.arm == 'rag') & (df.level == 'none')]
base = df[df.arm == 'baseline']
print(f'RAG {len(rag)} answers | baseline {len(base)} answers')
summary = pd.concat([rag.assign(arm='RAG'), base.assign(arm='no-RAG')]).groupby('arm').agg(
 faithfulness=('faithfulness','mean'), scored=('faithfulness','count'),
 citation_precision=('citation_precision','mean'), relevancy=('relevancy','mean'),
 fkgl=('fkgl','mean'), words=('words','mean'), refused=('refused','sum')).round(3)
print(summary)
print('\nlowest faithfulness, baseline:')
print(base.nsmallest(5,'faithfulness')
[['question','faithfulness','words']].to_string(index=False))
print('\nbaseline faithfulness by relevancy score:')
print(base.groupby('relevancy').agg(faithfulness=('faithfulness','mean'),
 words=('words','mean'), n=('faithfulness','size')).round(3))

RAG 115 answers | baseline 115 answers
        faithfulness  scored  citation_precision  relevancy    fkgl    words  \
arm                                                                            
RAG            0.989      92               0.933      3.939  10.631   87.235   
no-RAG         0.825     115                 NaN      4.348  10.344  187.000   

        refused  
arm              
RAG          23  
no-RAG        0  

lowest faithfulness, baseline:
                                                                  question  faithfulness  words
                                             What is (are) Stomach Cancer?          0.22    163
what research (or clinical trials) is being done for Swallowing Disorders?          0.40    112
                                                      How to prevent COPD?          0.44    146
                            What are the treatments for Myasthenia Gravis?          0.47    274
                            Do you have information abou